# Hybridisation functions of a four-dimensional system (archived)

Exploratory plots of the hybridisation functions $R_{jk}(\omega)$ and the quantities built from
them, for a 4×1 system (four beads, x only; the 2×2 alternative is left commented out in each
cell): the cavity and mechanical susceptibilities; the heterodyne spectrum (and the $x_1$ PSD)
next to an animation of every $R_{jk}(\omega)$ in the complex plane, for couplings
$g/2\pi$ = −5 … −40 kHz and two Ω models; grids of all $R_{jk}$; the deranged determinant; the
diagonal and off-diagonal adjugate entries; and all solutions $h_{jk}$. The power-series curves
are evaluated at orders 1-3 against the exact (order 4) result.

**Runtime:** a couple of minutes with `MAKE_ANIMATIONS = False`, where each animation is replaced
by a static preview (every 10th frame drawn, so the trails show) and the 0.1 kHz
spectrum grid is coarsened to 0.5 kHz. `MAKE_ANIMATIONS = True` writes
17 animations of 101 frames to `output/`.

**Updated Nov 2026** to current levisim:

- Ω model `'uniform'` → `'equal spacing'`: the same model under its current name (identical
  parameters Ω_mid, Ω_span and the same evenly spaced frequencies), renamed in Oct 2025;
- `calculate_heterodyne_spectrum` → `heterodyne_spectrum`;
  `calculate_all_hybridised_position_spectra(method)` → `all_position_power_spectra(method,
  zpf_scaled=False)` (m²/Hz, as before);
- `deranged_determinant_terms` → `deranged_determinant_by_order[evaluation_order]`,
  `deranged_determinant_value` → `deranged_determinant`, `deranged_adjugate_entry_value` →
  `deranged_adjugate_matrix_entry`, `Dimension.χM` → `Dimension.χ`;
- the 2024 `'model': 'pairs'` keys `first_Ωx`/`Ω_gap` are written as `Ω_first`,
  `Ω_interpair_gap` and `Ω_intrapair_gap = Ω_gap/10` (the same frequencies), and
  `setup_hybridisation_power_series=True` → `load_in_power_series=True`;
- the removed colour list `forty_five_colours` → `levisim.core.colours.all_colours_list`, repeated;
- the off-diagonal adjugate cell now computes the determinant `det` itself (it used to rely on
  a later cell having been run first);
- animations are written to `output/` by `save_animation` instead of embedded as HTML.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.animation import writers

from levisim.core.plotting import set_plt_rcParams_to_xmgrace_style

# MAKE_ANIMATIONS = True computes every frame at full resolution and writes the animations to
# output/ (mp4 with ffmpeg, else gif). False (the smoke-test setting) draws a quick static preview
MAKE_ANIMATIONS = False
OUTPUT_DIR = Path('output')

set_plt_rcParams_to_xmgrace_style()
# curves touch the left and right edges of the axes
plt.rcParams['axes.xmargin'] = 0


def save_animation(animation, name):
    """Writes `animation` to output/<name>.mp4 (or .gif without ffmpeg), never into the notebook"""
    OUTPUT_DIR.mkdir(exist_ok=True)
    writer = 'ffmpeg' if writers.is_available('ffmpeg') else 'pillow'
    path = OUTPUT_DIR / f'{name}.{"mp4" if writer == "ffmpeg" else "gif"}'
    animation.save(path, writer=writer)
    print(f'Wrote {path}')
    return path


def preview_animation(update, num_frames, every=10):
    """Static preview of a FuncAnimation: draws every `every`-th frame onto its figure, so the trails show"""
    for frame in range(0, num_frames, every):
        update(frame)

# All susceptibilities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from functools import partial
from levisim import *

default_g = -40*kHz
toy_params = {
    # 'Ω_model': 'pairs',
    # 'first_Ωx': 312*kHz,
    # 'Ω_gap': 15*kHz,
    # 'Ωz_to_Ωx_multiplier': 10,

    'Ω_model': 'equal spacing',  # called 'uniform' before Oct 2025
    'Ω_mid': 300*kHz,
    'Ω_span': 90*kHz,

    'toy_gx': default_g,
    'toy_gy': default_g,

    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'κ': 400*kHz,
    # 'L': 0.96*cm,
    # 'Finesse': 40_000,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
N, ax = 1, 3
N, ax = 2, 2
N, ax = 4, 1
ω_low, ω_high, ω_step = 0, 400, 0.5
multitrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=ax,
    ω_range=np.arange(ω_low, ω_high, ω_step) * kHz,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    load_in_power_series=False,
)

omega_range = np.arange(-1000, 1000, 0.5) * kHz

colours = plt.rcParams['axes.prop_cycle'].by_key()['color']
ax_key_to_func = {'Re': np.real, 'Im': np.imag, 'Abs': np.abs}
ax_key_to_colours = {'Re': colours[0], 'Im': colours[1], 'Abs': colours[2]}

def plot_ReImAbs(
    plot_func,
    plot_func_str,
    yscale_dict=None,
):
    """
    Plots real, imaginary and absolute parts of a given complex function
    """
    if yscale_dict is None:
        yscale_dict = {'Re': 'linear', 'Im': 'linear', 'Abs': 'linear'}

    fig = plt.figure(layout='constrained')
    ax_dict = fig.subplot_mosaic([
        ['Re'],
        ['Im'],
        ['Abs'],
    ])
    for ax_name, ax in ax_dict.items():
        ReImAbs = ax_key_to_func[ax_name]
        ax.plot(
            omega_range / kHz,
            [ReImAbs(plot_func(ω=ω)) for ω in omega_range],
            label=f'{ax_name}[{plot_func_str}]',
            color=ax_key_to_colours[ax_name],
        )
        ax.grid(True)
        ax.legend()
        ax.set_yscale(yscale_dict[ax_name])
        if ax_name == 'Abs':
            ax.set_xlabel('Frequency (kHz)')
    plt.show()

plot_ReImAbs(multitrap.cavity.χC, 'χC')
plot_ReImAbs(multitrap.cavity.ηm, 'η-')
plot_ReImAbs(multitrap.cavity.ηp, 'η+')

g = -1*kHz
def μm_ηm_g2(dim, ω):
    """
    Rjk (with uniform, ampltiude coupled g) end up taking the form
    z / (1 + z)
    where
    z = μjn- × ηC- × g2
    """
    return dim.μm(ω) * multitrap.cavity.ηm(ω) * g**2

for dim in list(multitrap.dimensions.values())[:2]:
    jn = dim.label
    plot_ReImAbs(dim.χ, f'χ{jn}', yscale_dict={'Re': 'log', 'Im': 'symlog', 'Abs': 'log'})
    plot_ReImAbs(dim.μm, f'μ{jn}-', yscale_dict={'Re': 'symlog', 'Im': 'symlog', 'Abs': 'log'})
    plot_ReImAbs(dim.μp, f'μ{jn}+', yscale_dict={'Re': 'log', 'Im': 'symlog', 'Abs': 'log'})

    plot_ReImAbs(partial(μm_ηm_g2, dim=dim), f'μ{jn}.ηC.g2', yscale_dict={'Re': 'symlog', 'Im': 'symlog', 'Abs': 'log'})


all_partial_Rjk = [partial(multitrap.hybridisation_term_value, term=Rjk) for Rjk in multitrap.all_Rjk]
all_jk = [Rjk.pairs for Rjk in multitrap.all_Rjk]
for ii, Rjk in enumerate(all_partial_Rjk[::3]):
    jn, km = all_jk[ii][0]
    plot_ReImAbs(Rjk, f'R{jn}{km}')

# Spectra vs. all Rjk at different g

##  Ω_model = pairs

### g/2π = -5 kHz

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from levisim import *


default_g = -5*kHz
toy_params = {
    'Ω_model': 'pairs',
    'toy_gx': default_g,
    'toy_gy': default_g,
    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'κ': 400*kHz,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
num_traps, Cartesian_axes_per_trap = 2, 2
num_traps, Cartesian_axes_per_trap = 4, 1
ω_range = np.arange(250, 350, 0.1 if MAKE_ANIMATIONS else 0.5) * kHz
multitrap = MultiTrap(
    num_traps=num_traps,
    Cartesian_axes_per_trap=Cartesian_axes_per_trap,
    ω_range=ω_range,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    generate_power_series=True,
)

multitrap.toy_params_info

fig = plt.figure(layout='constrained', figsize=(10, 4), dpi=150)
ax_dict = fig.subplot_mosaic([
    ['PSD', 'R1'],
])

psd_ax = ax_dict['PSD']
psd_colours = discrete_viridis[4]

het = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)
multitrap.evaluation_order = 3
het_3 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 2
het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 1
het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
psd_ax.plot(ω_range / kHz, het_1, label='(1st order)', color=psd_colours[0])
psd_ax.plot(ω_range / kHz, het_2, label='(2nd order)', color=psd_colours[1])
psd_ax.plot(ω_range / kHz, het_3, label='(3rd order)', color=psd_colours[2])
psd_ax.plot(ω_range / kHz, het, label='Heterodyne', color=psd_colours[3])

# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION, zpf_scaled=False)
# psd = psd_dict['x1']
# multitrap.evaluation_order = 3
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_3 = psd_dict['x1']
# multitrap.evaluation_order = 2
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_2 = psd_dict['x1']
# multitrap.evaluation_order = 1
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_1 = psd_dict['x1']
# het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# multitrap.evaluation_order = 1
# het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# psd_ax.plot(ω_range / kHz, psd_1, label='(1st order)', color=psd_colours[0])
# psd_ax.plot(ω_range / kHz, psd_2, label='(2nd order)', color=psd_colours[1])
# psd_ax.plot(ω_range / kHz, psd_3, label='(3rd order)', color=psd_colours[2])
# psd_ax.plot(ω_range / kHz, psd, label='x1 PSD', color=psd_colours[3])

vline = psd_ax.axvline(x=ω_range[0] / kHz, ymin=1e-2, ymax=1e2, color='r')
psd_ax.set_xlabel('Frequency (kHz)')
psd_ax.set_yscale('log')
psd_ax.legend()
psd_ax.grid(True, linestyle='--', alpha=0.7)

terms = multitrap.all_Rjk
N = num_traps * Cartesian_axes_per_trap
assert len(terms) == N**2 - N
print(f'{len(terms)=}')
print(f'{[term_string(term) for term in terms]}')

ax = ax_dict['R1']
lim = 2
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect('equal', 'box')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, linestyle='--', alpha=0.7)

circle = matplotlib.patches.Circle((0, 0), radius=1, color='k', alpha=0.3)
ax.add_patch(circle)

colours = (all_colours_list * 3)[:45]
arrows = [
    ax.plot([], [], color=colour, marker='o')
    for colour in colours
]
arrows = sum(arrows, [])

output_trails = [
    ax.scatter([], [], color=colour, alpha=0.5, s=10)
    for colour in colours
]

trails = [[] for colour in colours]

num_terms = len(terms)
arrows = arrows[:num_terms]
output_trails = output_trails[:num_terms]
trails = trails[:num_terms]
terms = terms[:num_terms]
ω_input = np.linspace(250, 350, 100 + 1, endpoint=True) * kHz

def update(frame):
    ω = ω_input[frame]

    vline.set_xdata([ω / kHz, ω / kHz])

    for ii, term in enumerate(terms):
        fω = multitrap.hybridisation_term_value(term, ω)

        # if np.abs(fω) >= 1:
        #     print(f'Abs[{term_string(term)}(ω/2π = {ω / kHz:.1f} kHz)] = {np.abs(fω):.2f} > 1')

        arrows[ii].set_label(f'{term_string(term)}(ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])

    ax.legend(loc='upper right')
    ax.legend(bbox_to_anchor=(1.5, 1), loc='upper right')
    psd_ax.set_title(f'Red line at ω/2π={ω / kHz:.1f} kHz\n(g/2π = {default_g / kHz:.1f} kHz)')

    return (*arrows, *output_trails)

ani = FuncAnimation(fig, update, frames=len(ω_input), interval=2, blit=True)
save_name = f'{N}D {num_traps}x{Cartesian_axes_per_trap} - Heterodyne vs Rjk at g={default_g / kHz:.0f} kHz.mp4'
if MAKE_ANIMATIONS:
    save_animation(ani, save_name.removesuffix('.mp4'))
else:
    preview_animation(update, len(ω_input))

### g/2π = -10 kHz

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from levisim import *


default_g = -10*kHz
toy_params = {
    'Ω_model': 'pairs',
    'toy_gx': default_g,
    'toy_gy': default_g,
    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'κ': 400*kHz,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
num_traps, Cartesian_axes_per_trap = 2, 2
num_traps, Cartesian_axes_per_trap = 4, 1
ω_range = np.arange(250, 350, 0.1 if MAKE_ANIMATIONS else 0.5) * kHz
multitrap = MultiTrap(
    num_traps=num_traps,
    Cartesian_axes_per_trap=Cartesian_axes_per_trap,
    ω_range=ω_range,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    generate_power_series=True,
)

multitrap.toy_params_info

fig = plt.figure(layout='constrained', figsize=(10, 4), dpi=150)
ax_dict = fig.subplot_mosaic([
    ['PSD', 'R1'],
])

psd_ax = ax_dict['PSD']
psd_colours = discrete_viridis[4]

het = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)
multitrap.evaluation_order = 3
het_3 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 2
het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 1
het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
psd_ax.plot(ω_range / kHz, het_1, label='(1st order)', color=psd_colours[0])
psd_ax.plot(ω_range / kHz, het_2, label='(2nd order)', color=psd_colours[1])
psd_ax.plot(ω_range / kHz, het_3, label='(3rd order)', color=psd_colours[2])
psd_ax.plot(ω_range / kHz, het, label='Heterodyne', color=psd_colours[3])

# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION, zpf_scaled=False)
# psd = psd_dict['x1']
# multitrap.evaluation_order = 3
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_3 = psd_dict['x1']
# multitrap.evaluation_order = 2
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_2 = psd_dict['x1']
# multitrap.evaluation_order = 1
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_1 = psd_dict['x1']
# het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# multitrap.evaluation_order = 1
# het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# psd_ax.plot(ω_range / kHz, psd_1, label='(1st order)', color=psd_colours[0])
# psd_ax.plot(ω_range / kHz, psd_2, label='(2nd order)', color=psd_colours[1])
# psd_ax.plot(ω_range / kHz, psd_3, label='(3rd order)', color=psd_colours[2])
# psd_ax.plot(ω_range / kHz, psd, label='x1 PSD', color=psd_colours[3])

vline = psd_ax.axvline(x=ω_range[0] / kHz, ymin=1e-2, ymax=1e2, color='r')
psd_ax.set_xlabel('Frequency (kHz)')
psd_ax.set_yscale('log')
psd_ax.legend()
psd_ax.grid(True, linestyle='--', alpha=0.7)

terms = multitrap.all_Rjk
N = num_traps * Cartesian_axes_per_trap
assert len(terms) == N**2 - N
print(f'{len(terms)=}')
print(f'{[term_string(term) for term in terms]}')

ax = ax_dict['R1']
lim = 2
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect('equal', 'box')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, linestyle='--', alpha=0.7)

circle = matplotlib.patches.Circle((0, 0), radius=1, color='k', alpha=0.3)
ax.add_patch(circle)

colours = (all_colours_list * 3)[:45]
arrows = [
    ax.plot([], [], color=colour, marker='o')
    for colour in colours
]
arrows = sum(arrows, [])

output_trails = [
    ax.scatter([], [], color=colour, alpha=0.5, s=10)
    for colour in colours
]

trails = [[] for colour in colours]

num_terms = len(terms)
arrows = arrows[:num_terms]
output_trails = output_trails[:num_terms]
trails = trails[:num_terms]
terms = terms[:num_terms]
ω_input = np.linspace(250, 350, 100 + 1, endpoint=True) * kHz

def update(frame):
    ω = ω_input[frame]

    vline.set_xdata([ω / kHz, ω / kHz])

    for ii, term in enumerate(terms):
        fω = multitrap.hybridisation_term_value(term, ω)

        # if np.abs(fω) >= 1:
        #     print(f'Abs[{term_string(term)}(ω/2π = {ω / kHz:.1f} kHz)] = {np.abs(fω):.2f} > 1')

        arrows[ii].set_label(f'{term_string(term)}(ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])

    ax.legend(loc='upper right')
    ax.legend(bbox_to_anchor=(1.5, 1), loc='upper right')
    psd_ax.set_title(f'Red line at ω/2π={ω / kHz:.1f} kHz\n(g/2π = {default_g / kHz:.1f} kHz)')

    return (*arrows, *output_trails)

ani = FuncAnimation(fig, update, frames=len(ω_input), interval=2, blit=True)
save_name = f'{N}D {num_traps}x{Cartesian_axes_per_trap} - Heterodyne vs Rjk at g={default_g / kHz:.0f} kHz.mp4'
if MAKE_ANIMATIONS:
    save_animation(ani, save_name.removesuffix('.mp4'))
else:
    preview_animation(update, len(ω_input))

### g/2π = -15 kHz

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from levisim import *


default_g = -15*kHz
toy_params = {
    'Ω_model': 'pairs',
    'toy_gx': default_g,
    'toy_gy': default_g,
    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'κ': 400*kHz,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
num_traps, Cartesian_axes_per_trap = 2, 2
num_traps, Cartesian_axes_per_trap = 4, 1
ω_range = np.arange(250, 350, 0.1 if MAKE_ANIMATIONS else 0.5) * kHz
multitrap = MultiTrap(
    num_traps=num_traps,
    Cartesian_axes_per_trap=Cartesian_axes_per_trap,
    ω_range=ω_range,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    generate_power_series=True,
)

multitrap.toy_params_info

fig = plt.figure(layout='constrained', figsize=(10, 4), dpi=150)
ax_dict = fig.subplot_mosaic([
    ['PSD', 'R1'],
])

psd_ax = ax_dict['PSD']
psd_colours = discrete_viridis[4]

het = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)
multitrap.evaluation_order = 3
het_3 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 2
het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 1
het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
psd_ax.plot(ω_range / kHz, het_1, label='(1st order)', color=psd_colours[0])
psd_ax.plot(ω_range / kHz, het_2, label='(2nd order)', color=psd_colours[1])
psd_ax.plot(ω_range / kHz, het_3, label='(3rd order)', color=psd_colours[2])
psd_ax.plot(ω_range / kHz, het, label='Heterodyne', color=psd_colours[3])

# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION, zpf_scaled=False)
# psd = psd_dict['x1']
# multitrap.evaluation_order = 3
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_3 = psd_dict['x1']
# multitrap.evaluation_order = 2
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_2 = psd_dict['x1']
# multitrap.evaluation_order = 1
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_1 = psd_dict['x1']
# het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# multitrap.evaluation_order = 1
# het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# psd_ax.plot(ω_range / kHz, psd_1, label='(1st order)', color=psd_colours[0])
# psd_ax.plot(ω_range / kHz, psd_2, label='(2nd order)', color=psd_colours[1])
# psd_ax.plot(ω_range / kHz, psd_3, label='(3rd order)', color=psd_colours[2])
# psd_ax.plot(ω_range / kHz, psd, label='x1 PSD', color=psd_colours[3])

vline = psd_ax.axvline(x=ω_range[0] / kHz, ymin=1e-2, ymax=1e2, color='r')
psd_ax.set_xlabel('Frequency (kHz)')
psd_ax.set_yscale('log')
psd_ax.legend()
psd_ax.grid(True, linestyle='--', alpha=0.7)

terms = multitrap.all_Rjk
N = num_traps * Cartesian_axes_per_trap
assert len(terms) == N**2 - N
print(f'{len(terms)=}')
print(f'{[term_string(term) for term in terms]}')

ax = ax_dict['R1']
lim = 2
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect('equal', 'box')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, linestyle='--', alpha=0.7)

circle = matplotlib.patches.Circle((0, 0), radius=1, color='k', alpha=0.3)
ax.add_patch(circle)

colours = (all_colours_list * 3)[:45]
arrows = [
    ax.plot([], [], color=colour, marker='o')
    for colour in colours
]
arrows = sum(arrows, [])

output_trails = [
    ax.scatter([], [], color=colour, alpha=0.5, s=10)
    for colour in colours
]

trails = [[] for colour in colours]

num_terms = len(terms)
arrows = arrows[:num_terms]
output_trails = output_trails[:num_terms]
trails = trails[:num_terms]
terms = terms[:num_terms]
ω_input = np.linspace(250, 350, 100 + 1, endpoint=True) * kHz

def update(frame):
    ω = ω_input[frame]

    vline.set_xdata([ω / kHz, ω / kHz])

    for ii, term in enumerate(terms):
        fω = multitrap.hybridisation_term_value(term, ω)

        # if np.abs(fω) >= 1:
        #     print(f'Abs[{term_string(term)}(ω/2π = {ω / kHz:.1f} kHz)] = {np.abs(fω):.2f} > 1')

        arrows[ii].set_label(f'{term_string(term)}(ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])

    ax.legend(loc='upper right')
    ax.legend(bbox_to_anchor=(1.5, 1), loc='upper right')
    psd_ax.set_title(f'Red line at ω/2π={ω / kHz:.1f} kHz\n(g/2π = {default_g / kHz:.1f} kHz)')

    return (*arrows, *output_trails)

ani = FuncAnimation(fig, update, frames=len(ω_input), interval=2, blit=True)
save_name = f'{N}D {num_traps}x{Cartesian_axes_per_trap} - Heterodyne vs Rjk at g={default_g / kHz:.0f} kHz.mp4'
if MAKE_ANIMATIONS:
    save_animation(ani, save_name.removesuffix('.mp4'))
else:
    preview_animation(update, len(ω_input))

### g/2π = -20 kHz

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from levisim import *


default_g = -20*kHz
toy_params = {
    'Ω_model': 'pairs',
    'toy_gx': default_g,
    'toy_gy': default_g,
    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'κ': 400*kHz,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
num_traps, Cartesian_axes_per_trap = 2, 2
num_traps, Cartesian_axes_per_trap = 4, 1
ω_range = np.arange(250, 350, 0.1 if MAKE_ANIMATIONS else 0.5) * kHz
multitrap = MultiTrap(
    num_traps=num_traps,
    Cartesian_axes_per_trap=Cartesian_axes_per_trap,
    ω_range=ω_range,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    generate_power_series=True,
)

multitrap.toy_params_info

fig = plt.figure(layout='constrained', figsize=(10, 4), dpi=150)
ax_dict = fig.subplot_mosaic([
    ['PSD', 'R1'],
])

psd_ax = ax_dict['PSD']
psd_colours = discrete_viridis[4]

het = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)
multitrap.evaluation_order = 3
het_3 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 2
het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 1
het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
psd_ax.plot(ω_range / kHz, het_1, label='(1st order)', color=psd_colours[0])
psd_ax.plot(ω_range / kHz, het_2, label='(2nd order)', color=psd_colours[1])
psd_ax.plot(ω_range / kHz, het_3, label='(3rd order)', color=psd_colours[2])
psd_ax.plot(ω_range / kHz, het, label='Heterodyne', color=psd_colours[3])

# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION, zpf_scaled=False)
# psd = psd_dict['x1']
# multitrap.evaluation_order = 3
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_3 = psd_dict['x1']
# multitrap.evaluation_order = 2
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_2 = psd_dict['x1']
# multitrap.evaluation_order = 1
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_1 = psd_dict['x1']
# het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# multitrap.evaluation_order = 1
# het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# psd_ax.plot(ω_range / kHz, psd_1, label='(1st order)', color=psd_colours[0])
# psd_ax.plot(ω_range / kHz, psd_2, label='(2nd order)', color=psd_colours[1])
# psd_ax.plot(ω_range / kHz, psd_3, label='(3rd order)', color=psd_colours[2])
# psd_ax.plot(ω_range / kHz, psd, label='x1 PSD', color=psd_colours[3])

vline = psd_ax.axvline(x=ω_range[0] / kHz, ymin=1e-2, ymax=1e2, color='r')
psd_ax.set_xlabel('Frequency (kHz)')
psd_ax.set_yscale('log')
psd_ax.legend()
psd_ax.grid(True, linestyle='--', alpha=0.7)

terms = multitrap.all_Rjk
N = num_traps * Cartesian_axes_per_trap
assert len(terms) == N**2 - N
print(f'{len(terms)=}')
print(f'{[term_string(term) for term in terms]}')

ax = ax_dict['R1']
lim = 2
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect('equal', 'box')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, linestyle='--', alpha=0.7)

circle = matplotlib.patches.Circle((0, 0), radius=1, color='k', alpha=0.3)
ax.add_patch(circle)

colours = (all_colours_list * 3)[:45]
arrows = [
    ax.plot([], [], color=colour, marker='o')
    for colour in colours
]
arrows = sum(arrows, [])

output_trails = [
    ax.scatter([], [], color=colour, alpha=0.5, s=10)
    for colour in colours
]

trails = [[] for colour in colours]

num_terms = len(terms)
arrows = arrows[:num_terms]
output_trails = output_trails[:num_terms]
trails = trails[:num_terms]
terms = terms[:num_terms]
ω_input = np.linspace(250, 350, 100 + 1, endpoint=True) * kHz

def update(frame):
    ω = ω_input[frame]

    vline.set_xdata([ω / kHz, ω / kHz])

    for ii, term in enumerate(terms):
        fω = multitrap.hybridisation_term_value(term, ω)

        # if np.abs(fω) >= 1:
        #     print(f'Abs[{term_string(term)}(ω/2π = {ω / kHz:.1f} kHz)] = {np.abs(fω):.2f} > 1')

        arrows[ii].set_label(f'{term_string(term)}(ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])

    ax.legend(loc='upper right')
    ax.legend(bbox_to_anchor=(1.5, 1), loc='upper right')
    psd_ax.set_title(f'Red line at ω/2π={ω / kHz:.1f} kHz\n(g/2π = {default_g / kHz:.1f} kHz)')

    return (*arrows, *output_trails)

ani = FuncAnimation(fig, update, frames=len(ω_input), interval=2, blit=True)
save_name = f'{N}D {num_traps}x{Cartesian_axes_per_trap} - Heterodyne vs Rjk at g={default_g / kHz:.0f} kHz.mp4'
if MAKE_ANIMATIONS:
    save_animation(ani, save_name.removesuffix('.mp4'))
else:
    preview_animation(update, len(ω_input))

### g/2π = -30 kHz

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from levisim import *


default_g = -30*kHz
toy_params = {
    'Ω_model': 'pairs',
    'toy_gx': default_g,
    'toy_gy': default_g,
    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'κ': 400*kHz,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
num_traps, Cartesian_axes_per_trap = 2, 2
num_traps, Cartesian_axes_per_trap = 4, 1
ω_range = np.arange(250, 350, 0.1 if MAKE_ANIMATIONS else 0.5) * kHz
multitrap = MultiTrap(
    num_traps=num_traps,
    Cartesian_axes_per_trap=Cartesian_axes_per_trap,
    ω_range=ω_range,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    generate_power_series=True,
)

multitrap.toy_params_info

fig = plt.figure(layout='constrained', figsize=(10, 4), dpi=150)
ax_dict = fig.subplot_mosaic([
    ['PSD', 'R1'],
])

psd_ax = ax_dict['PSD']
psd_colours = discrete_viridis[4]

het = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)
multitrap.evaluation_order = 3
het_3 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 2
het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 1
het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
psd_ax.plot(ω_range / kHz, het_1, label='(1st order)', color=psd_colours[0])
psd_ax.plot(ω_range / kHz, het_2, label='(2nd order)', color=psd_colours[1])
psd_ax.plot(ω_range / kHz, het_3, label='(3rd order)', color=psd_colours[2])
psd_ax.plot(ω_range / kHz, het, label='Heterodyne', color=psd_colours[3])

# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION, zpf_scaled=False)
# psd = psd_dict['x1']
# multitrap.evaluation_order = 3
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_3 = psd_dict['x1']
# multitrap.evaluation_order = 2
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_2 = psd_dict['x1']
# multitrap.evaluation_order = 1
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_1 = psd_dict['x1']
# het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# multitrap.evaluation_order = 1
# het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# psd_ax.plot(ω_range / kHz, psd_1, label='(1st order)', color=psd_colours[0])
# psd_ax.plot(ω_range / kHz, psd_2, label='(2nd order)', color=psd_colours[1])
# psd_ax.plot(ω_range / kHz, psd_3, label='(3rd order)', color=psd_colours[2])
# psd_ax.plot(ω_range / kHz, psd, label='x1 PSD', color=psd_colours[3])

vline = psd_ax.axvline(x=ω_range[0] / kHz, ymin=1e-2, ymax=1e2, color='r')
psd_ax.set_xlabel('Frequency (kHz)')
psd_ax.set_yscale('log')
psd_ax.legend()
psd_ax.grid(True, linestyle='--', alpha=0.7)

terms = multitrap.all_Rjk
N = num_traps * Cartesian_axes_per_trap
assert len(terms) == N**2 - N
print(f'{len(terms)=}')
print(f'{[term_string(term) for term in terms]}')

ax = ax_dict['R1']
lim = 2
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect('equal', 'box')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, linestyle='--', alpha=0.7)

circle = matplotlib.patches.Circle((0, 0), radius=1, color='k', alpha=0.3)
ax.add_patch(circle)

colours = (all_colours_list * 3)[:45]
arrows = [
    ax.plot([], [], color=colour, marker='o')
    for colour in colours
]
arrows = sum(arrows, [])

output_trails = [
    ax.scatter([], [], color=colour, alpha=0.5, s=10)
    for colour in colours
]

trails = [[] for colour in colours]

num_terms = len(terms)
arrows = arrows[:num_terms]
output_trails = output_trails[:num_terms]
trails = trails[:num_terms]
terms = terms[:num_terms]
ω_input = np.linspace(250, 350, 100 + 1, endpoint=True) * kHz

def update(frame):
    ω = ω_input[frame]

    vline.set_xdata([ω / kHz, ω / kHz])

    for ii, term in enumerate(terms):
        fω = multitrap.hybridisation_term_value(term, ω)

        # if np.abs(fω) >= 1:
        #     print(f'Abs[{term_string(term)}(ω/2π = {ω / kHz:.1f} kHz)] = {np.abs(fω):.2f} > 1')

        arrows[ii].set_label(f'{term_string(term)}(ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])

    ax.legend(loc='upper right')
    ax.legend(bbox_to_anchor=(1.5, 1), loc='upper right')
    psd_ax.set_title(f'Red line at ω/2π={ω / kHz:.1f} kHz\n(g/2π = {default_g / kHz:.1f} kHz)')

    return (*arrows, *output_trails)

ani = FuncAnimation(fig, update, frames=len(ω_input), interval=2, blit=True)
save_name = f'{N}D {num_traps}x{Cartesian_axes_per_trap} - Heterodyne vs Rjk at g={default_g / kHz:.0f} kHz.mp4'
if MAKE_ANIMATIONS:
    save_animation(ani, save_name.removesuffix('.mp4'))
else:
    preview_animation(update, len(ω_input))

### g/2π = -40 kHz

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from levisim import *


default_g = -40*kHz
toy_params = {
    'Ω_model': 'pairs',
    'toy_gx': default_g,
    'toy_gy': default_g,
    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'κ': 400*kHz,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
num_traps, Cartesian_axes_per_trap = 2, 2
num_traps, Cartesian_axes_per_trap = 4, 1
ω_range = np.arange(250, 350, 0.1 if MAKE_ANIMATIONS else 0.5) * kHz
multitrap = MultiTrap(
    num_traps=num_traps,
    Cartesian_axes_per_trap=Cartesian_axes_per_trap,
    ω_range=ω_range,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    generate_power_series=True,
)

multitrap.toy_params_info

fig = plt.figure(layout='constrained', figsize=(10, 4), dpi=150)
ax_dict = fig.subplot_mosaic([
    ['PSD', 'R1'],
])

psd_ax = ax_dict['PSD']
psd_colours = discrete_viridis[4]

het = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)
multitrap.evaluation_order = 3
het_3 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 2
het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 1
het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
psd_ax.plot(ω_range / kHz, het_1, label='(1st order)', color=psd_colours[0])
psd_ax.plot(ω_range / kHz, het_2, label='(2nd order)', color=psd_colours[1])
psd_ax.plot(ω_range / kHz, het_3, label='(3rd order)', color=psd_colours[2])
psd_ax.plot(ω_range / kHz, het, label='Heterodyne', color=psd_colours[3])

# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION, zpf_scaled=False)
# psd = psd_dict['x1']
# multitrap.evaluation_order = 3
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_3 = psd_dict['x1']
# multitrap.evaluation_order = 2
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_2 = psd_dict['x1']
# multitrap.evaluation_order = 1
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_1 = psd_dict['x1']
# het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# multitrap.evaluation_order = 1
# het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# psd_ax.plot(ω_range / kHz, psd_1, label='(1st order)', color=psd_colours[0])
# psd_ax.plot(ω_range / kHz, psd_2, label='(2nd order)', color=psd_colours[1])
# psd_ax.plot(ω_range / kHz, psd_3, label='(3rd order)', color=psd_colours[2])
# psd_ax.plot(ω_range / kHz, psd, label='x1 PSD', color=psd_colours[3])

vline = psd_ax.axvline(x=ω_range[0] / kHz, ymin=1e-2, ymax=1e2, color='r')
psd_ax.set_xlabel('Frequency (kHz)')
psd_ax.set_yscale('log')
psd_ax.legend()
psd_ax.grid(True, linestyle='--', alpha=0.7)

terms = multitrap.all_Rjk
N = num_traps * Cartesian_axes_per_trap
assert len(terms) == N**2 - N
print(f'{len(terms)=}')
print(f'{[term_string(term) for term in terms]}')

ax = ax_dict['R1']
lim = 2
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect('equal', 'box')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, linestyle='--', alpha=0.7)

circle = matplotlib.patches.Circle((0, 0), radius=1, color='k', alpha=0.3)
ax.add_patch(circle)

colours = (all_colours_list * 3)[:45]
arrows = [
    ax.plot([], [], color=colour, marker='o')
    for colour in colours
]
arrows = sum(arrows, [])

output_trails = [
    ax.scatter([], [], color=colour, alpha=0.5, s=10)
    for colour in colours
]

trails = [[] for colour in colours]

num_terms = len(terms)
arrows = arrows[:num_terms]
output_trails = output_trails[:num_terms]
trails = trails[:num_terms]
terms = terms[:num_terms]
ω_input = np.linspace(250, 350, 100 + 1, endpoint=True) * kHz

def update(frame):
    ω = ω_input[frame]

    vline.set_xdata([ω / kHz, ω / kHz])

    for ii, term in enumerate(terms):
        fω = multitrap.hybridisation_term_value(term, ω)

        # if np.abs(fω) >= 1:
        #     print(f'Abs[{term_string(term)}(ω/2π = {ω / kHz:.1f} kHz)] = {np.abs(fω):.2f} > 1')

        arrows[ii].set_label(f'{term_string(term)}(ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])

    ax.legend(loc='upper right')
    ax.legend(bbox_to_anchor=(1.5, 1), loc='upper right')
    psd_ax.set_title(f'Red line at ω/2π={ω / kHz:.1f} kHz\n(g/2π = {default_g / kHz:.1f} kHz)')

    return (*arrows, *output_trails)

ani = FuncAnimation(fig, update, frames=len(ω_input), interval=2, blit=True)
save_name = f'{N}D {num_traps}x{Cartesian_axes_per_trap} - Heterodyne vs Rjk at g={default_g / kHz:.0f} kHz.mp4'
if MAKE_ANIMATIONS:
    save_animation(ani, save_name.removesuffix('.mp4'))
else:
    preview_animation(update, len(ω_input))

## Ω_model = equal spacing (called 'uniform' before Oct 2025)

### g/2π = -5 kHz

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from levisim import *


default_g = -5*kHz
toy_params = {
    'Ω_model': 'equal spacing',  # called 'uniform' before Oct 2025
    'Ω_span': 30*kHz,
    'toy_gx': default_g,
    'toy_gy': default_g,
    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'κ': 400*kHz,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
num_traps, Cartesian_axes_per_trap = 2, 2
num_traps, Cartesian_axes_per_trap = 4, 1
ω_range = np.arange(250, 350, 0.1 if MAKE_ANIMATIONS else 0.5) * kHz
multitrap = MultiTrap(
    num_traps=num_traps,
    Cartesian_axes_per_trap=Cartesian_axes_per_trap,
    ω_range=ω_range,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    generate_power_series=True,
)

multitrap.toy_params_info

fig = plt.figure(layout='constrained', figsize=(10, 4), dpi=150)
ax_dict = fig.subplot_mosaic([
    ['PSD', 'R1'],
])

psd_ax = ax_dict['PSD']
psd_colours = discrete_viridis[4]

het = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)
multitrap.evaluation_order = 3
het_3 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 2
het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 1
het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
psd_ax.plot(ω_range / kHz, het_1, label='(1st order)', color=psd_colours[0])
psd_ax.plot(ω_range / kHz, het_2, label='(2nd order)', color=psd_colours[1])
psd_ax.plot(ω_range / kHz, het_3, label='(3rd order)', color=psd_colours[2])
psd_ax.plot(ω_range / kHz, het, label='Heterodyne', color=psd_colours[3])

# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION, zpf_scaled=False)
# psd = psd_dict['x1']
# multitrap.evaluation_order = 3
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_3 = psd_dict['x1']
# multitrap.evaluation_order = 2
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_2 = psd_dict['x1']
# multitrap.evaluation_order = 1
# _, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
# psd_1 = psd_dict['x1']
# het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# multitrap.evaluation_order = 1
# het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# psd_ax.plot(ω_range / kHz, psd_1, label='(1st order)', color=psd_colours[0])
# psd_ax.plot(ω_range / kHz, psd_2, label='(2nd order)', color=psd_colours[1])
# psd_ax.plot(ω_range / kHz, psd_3, label='(3rd order)', color=psd_colours[2])
# psd_ax.plot(ω_range / kHz, psd, label='x1 PSD', color=psd_colours[3])

vline = psd_ax.axvline(x=ω_range[0] / kHz, ymin=1e-2, ymax=1e2, color='r')
psd_ax.set_xlabel('Frequency (kHz)')
psd_ax.set_yscale('log')
psd_ax.legend()
psd_ax.grid(True, linestyle='--', alpha=0.7)

terms = multitrap.all_Rjk
N = num_traps * Cartesian_axes_per_trap
assert len(terms) == N**2 - N
print(f'{len(terms)=}')
print(f'{[term_string(term) for term in terms]}')

ax = ax_dict['R1']
lim = 2
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect('equal', 'box')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, linestyle='--', alpha=0.7)

circle = matplotlib.patches.Circle((0, 0), radius=1, color='k', alpha=0.3)
ax.add_patch(circle)

colours = (all_colours_list * 3)[:45]
arrows = [
    ax.plot([], [], color=colour, marker='o')
    for colour in colours
]
arrows = sum(arrows, [])

output_trails = [
    ax.scatter([], [], color=colour, alpha=0.5, s=10)
    for colour in colours
]

trails = [[] for colour in colours]

num_terms = len(terms)
arrows = arrows[:num_terms]
output_trails = output_trails[:num_terms]
trails = trails[:num_terms]
terms = terms[:num_terms]
ω_input = np.linspace(250, 350, 100 + 1, endpoint=True) * kHz

def update(frame):
    ω = ω_input[frame]

    vline.set_xdata([ω / kHz, ω / kHz])

    for ii, term in enumerate(terms):
        fω = multitrap.hybridisation_term_value(term, ω)

        # if np.abs(fω) >= 1:
        #     print(f'Abs[{term_string(term)}(ω/2π = {ω / kHz:.1f} kHz)] = {np.abs(fω):.2f} > 1')

        arrows[ii].set_label(f'{term_string(term)}(ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])

    ax.legend(loc='upper right')
    ax.legend(bbox_to_anchor=(1.5, 1), loc='upper right')
    psd_ax.set_title(f'Red line at ω/2π={ω / kHz:.1f} kHz\n(g/2π = {default_g / kHz:.1f} kHz)')

    return (*arrows, *output_trails)

ani = FuncAnimation(fig, update, frames=len(ω_input), interval=2, blit=True)
save_name = f'{N}D {num_traps}x{Cartesian_axes_per_trap} - {toy_params["Ω_model"]} - Heterodyne vs Rjk at g={default_g / kHz:.0f} kHz.mp4'
if MAKE_ANIMATIONS:
    save_animation(ani, save_name.removesuffix('.mp4'))
else:
    preview_animation(update, len(ω_input))

## psd...

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from levisim import *


default_g = -15*kHz
toy_params = {
    # 'Ω_model': 'pairs',
    # 'first_Ωx': 312*kHz,
    # 'Ω_gap': 15*kHz,
    # 'Ωz_to_Ωx_multiplier': 10,

    'Ω_model': 'equal spacing',  # called 'uniform' before Oct 2025
    'Ω_mid': 300*kHz,
    'Ω_span': 30*kHz,

    'toy_gx': default_g,
    'toy_gy': default_g,

    # 'φs': {
    #     1: 0*degrees,
    #     2: 90*degrees,
    #     3: 180*degrees,
    #     4: 270*degrees,
    # },

    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'κ': 400*kHz,
    # 'L': 0.96*cm,
    # 'Finesse': 40_000,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
N, ax = 2, 2
N, ax = 4, 1
ω_range = np.arange(250, 350, 0.1 if MAKE_ANIMATIONS else 0.5) * kHz
multitrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=ax,
    ω_range=ω_range,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    generate_power_series=True,
)

multitrap.toy_params_info


# fig, ax = plt.subplots(figsize=(6, 6), dpi=200)
# fig, ax = plt.subplots()

fig = plt.figure(layout='constrained', figsize=(10, 4), dpi=150)
ax_dict = fig.subplot_mosaic([
    ['PSD', 'R1'],
])

psd_ax = ax_dict['PSD']

# het = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)
# multitrap.evaluation_order = 3
# het_3 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# multitrap.evaluation_order = 2
# het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# multitrap.evaluation_order = 1
# het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)

# spec_colours = discrete_viridis[4]

# psd_ax.plot(ω_range / kHz, het_1, label='(1st order)', color=spec_colours[0])
# psd_ax.plot(ω_range / kHz, het_2, label='(2nd order)', color=spec_colours[1])
# psd_ax.plot(ω_range / kHz, het_3, label='(3rd order)', color=spec_colours[2])
# psd_ax.plot(ω_range / kHz, het, label='Heterodyne', color=spec_colours[3])

_, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION, zpf_scaled=False)
psd = psd_dict['x1']

multitrap.evaluation_order = 3
_, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
psd_3 = psd_dict['x1']

multitrap.evaluation_order = 2
_, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
psd_2 = psd_dict['x1']

multitrap.evaluation_order = 1
_, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
psd_1 = psd_dict['x1']

het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 1
het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)

spec_colours = discrete_viridis[4]

psd_ax.plot(ω_range / kHz, psd_1, label='(1st order)', color=spec_colours[0])
psd_ax.plot(ω_range / kHz, psd_2, label='(2nd order)', color=spec_colours[1])
psd_ax.plot(ω_range / kHz, psd_3, label='(3rd order)', color=spec_colours[2])
psd_ax.plot(ω_range / kHz, psd, label='x1 PSD', color=spec_colours[3])

# psd_ax.plot(ω_range / kHz, hom, label='Homodyne')
vline = psd_ax.axvline(x=ω_range[0] / kHz, ymin=1e-2, ymax=1e2, color='r')
psd_ax.set_xlabel('Frequency (kHz)')
psd_ax.set_title('$S_{hom}$(ω)')
psd_ax.set_yscale('log')
psd_ax.legend()
psd_ax.grid(True, linestyle='--', alpha=0.7)

terms = multitrap.all_Rjk
NN = N * ax
assert len(terms) == NN**2 - NN

print(f'{len(terms)=}')
print(f'{[term_string(term) for term in terms]}')

ω_input = np.linspace(250, 350, 100 + 1, endpoint=True) * kHz

ax = ax_dict['R1']
lim = 2
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect('equal', 'box')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, linestyle='--', alpha=0.7)

circle = matplotlib.patches.Circle((0, 0), radius=1, color='k', alpha=0.3)
ax.add_patch(circle)

marker = 'o'
colours = (all_colours_list * 3)[:45]
arrows = [
    ax.plot([], [], color=colour, marker=marker)
    for colour in colours
]
arrows = sum(arrows, [])

output_trails = [
    ax.scatter([], [], color=colour, alpha=0.5, s=10)
    for colour in colours
]

trails = [[] for colour in colours]

num_terms = len(terms)
arrows = arrows[:num_terms]
output_trails = output_trails[:num_terms]
trails = trails[:num_terms]
terms = terms[:num_terms]

def update(frame):
    ω = ω_input[frame]

    vline.set_xdata([ω / kHz, ω / kHz])

    for ii, term in enumerate(terms):
        fω = multitrap.hybridisation_term_value(term, ω)

        # if np.abs(fω) >= 1:
        #     print(f'Abs[{term_string(term)}(ω/2π = {ω / kHz:.1f} kHz)] = {np.abs(fω):.2f} > 1')

        arrows[ii].set_label(f'{term_string(term)}(ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])

    ax.legend(loc='upper right')
    ax.legend(bbox_to_anchor=(1.5, 1), loc='upper right')
    psd_ax.set_title(f'Red line at ω/2π={ω / kHz:.1f} kHz\n(g/2π = {default_g / kHz:.1f} kHz)')

    return (*arrows, *output_trails)

# plt.tight_layout()
ani = FuncAnimation(fig, update, frames=len(ω_input), interval=2, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, f'x1 PSD vs Rjk (equal spacing) at g={default_g / kHz:.0f} kHz')
else:
    preview_animation(update, len(ω_input))

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from levisim import *


default_g = -30*kHz
toy_params = {
    # 'Ω_model': 'pairs',
    # 'first_Ωx': 312*kHz,
    # 'Ω_gap': 15*kHz,
    # 'Ωz_to_Ωx_multiplier': 10,

    'Ω_model': 'equal spacing',  # called 'uniform' before Oct 2025
    'Ω_mid': 300*kHz,
    'Ω_span': 30*kHz,

    'toy_gx': default_g,
    'toy_gy': default_g,

    # 'φs': {
    #     1: 0*degrees,
    #     2: 90*degrees,
    #     3: 180*degrees,
    #     4: 270*degrees,
    # },

    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'κ': 400*kHz,
    # 'L': 0.96*cm,
    # 'Finesse': 40_000,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
N, ax = 2, 2
N, ax = 4, 1
ω_range = np.arange(250, 350, 0.1 if MAKE_ANIMATIONS else 0.5) * kHz
multitrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=ax,
    ω_range=ω_range,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    generate_power_series=True,
)

multitrap.toy_params_info


# fig, ax = plt.subplots(figsize=(6, 6), dpi=200)
# fig, ax = plt.subplots()

fig = plt.figure(layout='constrained', figsize=(10, 4), dpi=150)
ax_dict = fig.subplot_mosaic([
    ['PSD', 'R1'],
])

psd_ax = ax_dict['PSD']

# het = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)
# multitrap.evaluation_order = 3
# het_3 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# multitrap.evaluation_order = 2
# het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
# multitrap.evaluation_order = 1
# het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)

# spec_colours = discrete_viridis[4]

# psd_ax.plot(ω_range / kHz, het_1, label='(1st order)', color=spec_colours[0])
# psd_ax.plot(ω_range / kHz, het_2, label='(2nd order)', color=spec_colours[1])
# psd_ax.plot(ω_range / kHz, het_3, label='(3rd order)', color=spec_colours[2])
# psd_ax.plot(ω_range / kHz, het, label='Heterodyne', color=spec_colours[3])

_, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION, zpf_scaled=False)
psd = psd_dict['x1']

multitrap.evaluation_order = 3
_, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
psd_3 = psd_dict['x1']

multitrap.evaluation_order = 2
_, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
psd_2 = psd_dict['x1']

multitrap.evaluation_order = 1
_, psd_dict = multitrap.all_position_power_spectra(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION, zpf_scaled=False)
psd_1 = psd_dict['x1']

het_2 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)
multitrap.evaluation_order = 1
het_1 = multitrap.heterodyne_spectrum(QuantumLinearTheoryMethod.POWER_SERIES_HYBRIDISATION)

spec_colours = discrete_viridis[4]

psd_ax.plot(ω_range / kHz, psd_1, label='(1st order)', color=spec_colours[0])
psd_ax.plot(ω_range / kHz, psd_2, label='(2nd order)', color=spec_colours[1])
psd_ax.plot(ω_range / kHz, psd_3, label='(3rd order)', color=spec_colours[2])
psd_ax.plot(ω_range / kHz, psd, label='x1 PSD', color=spec_colours[3])

# psd_ax.plot(ω_range / kHz, hom, label='Homodyne')
vline = psd_ax.axvline(x=ω_range[0] / kHz, ymin=1e-2, ymax=1e2, color='r')
psd_ax.set_xlabel('Frequency (kHz)')
psd_ax.set_title('$S_{hom}$(ω)')
psd_ax.set_yscale('log')
psd_ax.legend()
psd_ax.grid(True, linestyle='--', alpha=0.7)

terms = multitrap.all_Rjk
NN = N * ax
assert len(terms) == NN**2 - NN

print(f'{len(terms)=}')
print(f'{[term_string(term) for term in terms]}')

ω_input = np.linspace(250, 350, 100 + 1, endpoint=True) * kHz

ax = ax_dict['R1']
lim = 2
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect('equal', 'box')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, linestyle='--', alpha=0.7)

circle = matplotlib.patches.Circle((0, 0), radius=1, color='k', alpha=0.3)
ax.add_patch(circle)

marker = 'o'
colours = (all_colours_list * 3)[:45]
arrows = [
    ax.plot([], [], color=colour, marker=marker)
    for colour in colours
]
arrows = sum(arrows, [])

output_trails = [
    ax.scatter([], [], color=colour, alpha=0.5, s=10)
    for colour in colours
]

trails = [[] for colour in colours]

num_terms = len(terms)
arrows = arrows[:num_terms]
output_trails = output_trails[:num_terms]
trails = trails[:num_terms]
terms = terms[:num_terms]

def update(frame):
    ω = ω_input[frame]

    vline.set_xdata([ω / kHz, ω / kHz])

    for ii, term in enumerate(terms):
        fω = multitrap.hybridisation_term_value(term, ω)

        # if np.abs(fω) >= 1:
        #     print(f'Abs[{term_string(term)}(ω/2π = {ω / kHz:.1f} kHz)] = {np.abs(fω):.2f} > 1')

        arrows[ii].set_label(f'{term_string(term)}(ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])

    ax.legend(loc='upper right')
    ax.legend(bbox_to_anchor=(1.5, 1), loc='upper right')
    psd_ax.set_title(f'Red line at ω/2π={ω / kHz:.1f} kHz\n(g/2π = {default_g / kHz:.1f} kHz)')

    return (*arrows, *output_trails)

# plt.tight_layout()
ani = FuncAnimation(fig, update, frames=len(ω_input), interval=2, blit=True)
if MAKE_ANIMATIONS:
    save_animation(ani, f'x1 PSD vs Rjk (equal spacing) at g={default_g / kHz:.0f} kHz (2)')
else:
    preview_animation(update, len(ω_input))

# Grid: All $R_{jk}(\omega)$

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


toy_params = {
    # the 2024 keys 'model': 'pairs', first_Ωx = 312 kHz and Ω_gap = 15 kHz, in today's names
    'Ω_model': 'pairs',
    'Ω_first': 312*kHz,
    'Ω_interpair_gap': 15*kHz,
    'Ω_intrapair_gap': 1.5*kHz,
    'Ωz_to_Ωx_multiplier': 10,

    'toy_gx': -40*kHz,
    'toy_gy': -40*kHz,
    # 'toy_gz': +170*kHz,
    # 'gxy': -2*kHz,
    # 'gxz': -15*kHz,
    # 'gyz': -15*kHz,

    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'L': 0.96*cm,
    'Finesse': 40_000,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
N, ax = 2, 2
ω_low, ω_high, ω_step = 0, 400, 0.5
multitrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=ax,
    ω_range=np.arange(ω_low, ω_high, ω_step) * kHz,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    # the 2×2 power-series pickle is tracked in git
    load_in_power_series=True,
)
multitrap.info
multitrap.info_per_dimension

# def truncate_power_series(power_series, truncation_power):
#     """
#     Truncates a power_series (an iterable of HybridisationTerm's)
#     leaving only orders up to truncation_power
#     """
#     return list(filter(lambda term: len(term.pairs) <= truncation_power, power_series))

# def sieve_power_series(power_series, sieve_power):
#     """
#     Sieves a power_series (an iterable of HybridisationTerm's)
#     leaving only orders of sieve_power
#     """
#     return list(filter(lambda term: len(term.pairs) == sieve_power, power_series))

# def sieve_power_series_for_sequence(power_series, sieve_sequence):
#     """
#     Truncates a power_series (an iterable of HybridisationTerm's)
#     leaving only orders up to truncation_power
#     """
#     return list(filter(lambda term: [pair[0] for pair in term.pairs] == sieve_sequence, power_series))

# sieve_power = 4
# terms = sieve_power_series(multitrap.deranged_determinant_by_order[multitrap.evaluation_order], sieve_power)
# sieve_sequence = ['x1', 'x2', 'x3', 'x4']
# terms = sieve_power_series_for_sequence(multitrap.deranged_determinant_by_order[multitrap.evaluation_order], sieve_sequence)

terms = []
for dim_j in list(multitrap.dimensions.values()):
    for dim_k in list(multitrap.dimensions.values()):

        if dim_j == dim_k:
            continue

        jn, km = dim_j.label, dim_k.label
        Rjk = HybridisationTerm(sign=1, pairs=[(jn, km)])
        print(term_string(Rjk))
        terms.append(Rjk)

NN = N * ax
assert len(terms) == NN**2 - NN

print(f'{len(terms)=}')
print(f'{terms=}')

ω_input = np.linspace(200, 350, 75, endpoint=True) * kHz
# fig, ax = plt.subplots(figsize=(6, 6), dpi=200)

Rjk_fig = plt.figure(layout='constrained', figsize=(15, 20), dpi=300)
Rjk_ax_dict = Rjk_fig.subplot_mosaic([
    [term_string(term) for term in terms[0:3]],
    [term_string(term) for term in terms[3:6]],
    [term_string(term) for term in terms[6:9]],
    [term_string(term) for term in terms[9:12]],
])

marker = 'o'
colours = [
    "#FF5733", "#33FF57", "#3357FF", "#FF33A6", "#33FFF6",
    "#F6FF33", "#FF9D33", "#33FF9D", "#9D33FF", "#FF3333",
    "#33FF33", "#3333FF", "#FF5733", "#57FF33", "#5733FF",
    "#A6FF33", "#33A6FF", "#FF33F6", "#F633FF", "#33F6FF",
    "#9DFF33", "#FF339D", "#FF3339", "#39FF33", "#3339FF",
    "#33FFF9", "#FF33C1", "#C1FF33", "#33C1FF", "#FF9D57",
    "#57FF9D", "#9D57FF", "#FF579D", "#579DFF", "#33FF63",
    "#FF6333", "#6333FF", "#33FFA6", "#FFA633", "#A633FF",
    "#FF33FF", "#33FFFF", "#FFFF33", "#FF8C00", "#8C00FF",
    "#FF00A0", "#00A0FF", "#8CFF00", "#00FF8C", "#FF5732"
]
lim = 2
arrows = []
output_trails = []
trails = []
for ii, ax in enumerate(list(Rjk_ax_dict.values())):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal', 'box')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.grid(True, linestyle='--', alpha=0.7)

    arrows.append(ax.plot([], [], color=colours[ii], marker=marker))
    output_trails.append(ax.scatter([], [], color=colours[ii], alpha=0.5, s=10))
    trails.append([])

arrows = sum(arrows, [])

num_terms = len(terms)
arrows = arrows[:num_terms]
output_trails = output_trails[:num_terms]
trails = trails[:num_terms]
terms = terms[:num_terms]

# Animation function
def update(frame):
    ω = ω_input[frame]

    # for ii, term in enumerate(terms):
    for ii, ax in enumerate(list(Rjk_ax_dict.values())):
        term = terms[ii]
        fω = multitrap.hybridisation_term_value(term, ω)
        arrows[ii].set_label(f'{term_string(term)}(ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])
        ax.legend(loc='upper right')
        ax.set_title(f'ω={ω / kHz:.1f} kHz')

    return (Rjk_fig, *arrows, *output_trails)

# Create the animation
ani = FuncAnimation(Rjk_fig, update, frames=len(ω_input), interval=2, blit=True)

if MAKE_ANIMATIONS:
    save_animation(ani, f'grid of all Rjk')
else:
    preview_animation(update, len(ω_input))

In [ ]:
Rjk_ax_dict

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


toy_params = {
    # the 2024 keys 'model': 'pairs', first_Ωx = 312 kHz and Ω_gap = 15 kHz, in today's names
    'Ω_model': 'pairs',
    'Ω_first': 312*kHz,
    'Ω_interpair_gap': 15*kHz,
    'Ω_intrapair_gap': 1.5*kHz,
    'Ωz_to_Ωx_multiplier': 10,

    'toy_gx': -40*kHz,
    'toy_gy': -40*kHz,
    # 'toy_gz': +170*kHz,
    # 'gxy': -2*kHz,
    # 'gxz': -15*kHz,
    # 'gyz': -15*kHz,

    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'L': 0.96*cm,
    'Finesse': 40_000,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
N, ax = 2, 2
ω_low, ω_high, ω_step = 0, 400, 0.5
multitrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=ax,
    ω_range=np.arange(ω_low, ω_high, ω_step) * kHz,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    # the 2×2 power-series pickle is tracked in git
    load_in_power_series=True,
)
multitrap.info
multitrap.info_per_dimension

# def truncate_power_series(power_series, truncation_power):
#     """
#     Truncates a power_series (an iterable of HybridisationTerm's)
#     leaving only orders up to truncation_power
#     """
#     return list(filter(lambda term: len(term.pairs) <= truncation_power, power_series))

def sieve_power_series(power_series, sieve_power):
    """
    Sieves a power_series (an iterable of HybridisationTerm's)
    leaving only orders of sieve_power
    """
    return list(filter(lambda term: len(term.pairs) == sieve_power, power_series))

# def sieve_power_series_for_sequence(power_series, sieve_sequence):
#     """
#     Truncates a power_series (an iterable of HybridisationTerm's)
#     leaving only orders up to truncation_power
#     """
#     return list(filter(lambda term: [pair[0] for pair in term.pairs] == sieve_sequence, power_series))

sieve_power = 2
terms = sieve_power_series(multitrap.deranged_determinant_by_order[multitrap.evaluation_order], sieve_power)
# sieve_sequence = ['x1', 'x2', 'x3', 'x4']
# terms = sieve_power_series_for_sequence(multitrap.deranged_determinant_by_order[multitrap.evaluation_order], sieve_sequence)

# terms = []
# for dim_j in list(multitrap.dimensions.values()):
#     for dim_k in list(multitrap.dimensions.values()):

#         if dim_j == dim_k:
#             continue

#         jn, km = dim_j.label, dim_k.label
#         Rjk = HybridisationTerm(sign=1, pairs=[(jn, km)])
#         print(term_string(Rjk))
#         terms.append(Rjk)

# NN = N * ax
# assert len(terms) == NN**2 - NN

assert len(terms) == 6

print(f'{len(terms)=}')
print(f'{terms=}')

ω_input = np.linspace(200, 350, 150, endpoint=True) * kHz
# fig, ax = plt.subplots(figsize=(6, 6), dpi=200)

Rjk_fig = plt.figure(layout='constrained', figsize=(15, 10), dpi=200)
Rjk_ax_dict = Rjk_fig.subplot_mosaic([
    [term_string(term) for term in terms[0:3]],
    [term_string(term) for term in terms[3:6]],
])

marker = 'o'
colours = [
    "#FF5733", "#33FF57", "#3357FF", "#FF33A6", "#33FFF6",
    "#F6FF33", "#FF9D33", "#33FF9D", "#9D33FF", "#FF3333",
    "#33FF33", "#3333FF", "#FF5733", "#57FF33", "#5733FF",
    "#A6FF33", "#33A6FF", "#FF33F6", "#F633FF", "#33F6FF",
    "#9DFF33", "#FF339D", "#FF3339", "#39FF33", "#3339FF",
    "#33FFF9", "#FF33C1", "#C1FF33", "#33C1FF", "#FF9D57",
    "#57FF9D", "#9D57FF", "#FF579D", "#579DFF", "#33FF63",
    "#FF6333", "#6333FF", "#33FFA6", "#FFA633", "#A633FF",
    "#FF33FF", "#33FFFF", "#FFFF33", "#FF8C00", "#8C00FF",
    "#FF00A0", "#00A0FF", "#8CFF00", "#00FF8C", "#FF5732"
]
lim = 2
arrows = []
output_trails = []
trails = []
for ii, ax in enumerate(list(Rjk_ax_dict.values())):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal', 'box')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.grid(True, linestyle='--', alpha=0.7)

    arrows.append(ax.plot([], [], color=colours[ii], marker=marker))
    output_trails.append(ax.scatter([], [], color=colours[ii], alpha=0.5, s=10))
    trails.append([])

arrows = sum(arrows, [])

num_terms = len(terms)
arrows = arrows[:num_terms]
output_trails = output_trails[:num_terms]
trails = trails[:num_terms]
terms = terms[:num_terms]

# Animation function
def update(frame):
    ω = ω_input[frame]

    # for ii, term in enumerate(terms):
    for ii, ax in enumerate(list(Rjk_ax_dict.values())):
        term = terms[ii]
        fω = multitrap.hybridisation_term_value(term, ω)
        arrows[ii].set_label(f'{term_string(term)}(ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])
        ax.legend(loc='upper right')
        ax.set_title(f'ω={ω / kHz:.1f} kHz')

    return (Rjk_fig, *arrows, *output_trails)

# Create the animation
ani = FuncAnimation(Rjk_fig, update, frames=len(ω_input), interval=2, blit=False)

if MAKE_ANIMATIONS:
    save_animation(ani, f'grid of all Rjk (2)')
else:
    preview_animation(update, len(ω_input))

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


toy_params = {
    # the 2024 keys 'model': 'pairs', first_Ωx = 312 kHz and Ω_gap = 15 kHz, in today's names
    'Ω_model': 'pairs',
    'Ω_first': 312*kHz,
    'Ω_interpair_gap': 15*kHz,
    'Ω_intrapair_gap': 1.5*kHz,
    'Ωz_to_Ωx_multiplier': 10,

    'toy_gx': -40*kHz,
    'toy_gy': -40*kHz,
    # 'toy_gz': +170*kHz,
    # 'gxy': -2*kHz,
    # 'gxz': -15*kHz,
    # 'gyz': -15*kHz,

    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'L': 0.96*cm,
    'Finesse': 40_000,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
N, ax = 2, 2
ω_low, ω_high, ω_step = 0, 400, 0.5
multitrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=ax,
    ω_range=np.arange(ω_low, ω_high, ω_step) * kHz,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    # the 2×2 power-series pickle is tracked in git
    load_in_power_series=True,
)
multitrap.info
multitrap.info_per_dimension

# def truncate_power_series(power_series, truncation_power):
#     """
#     Truncates a power_series (an iterable of HybridisationTerm's)
#     leaving only orders up to truncation_power
#     """
#     return list(filter(lambda term: len(term.pairs) <= truncation_power, power_series))

def sieve_power_series(power_series, sieve_power):
    """
    Sieves a power_series (an iterable of HybridisationTerm's)
    leaving only orders of sieve_power
    """
    return list(filter(lambda term: len(term.pairs) == sieve_power, power_series))

# def sieve_power_series_for_sequence(power_series, sieve_sequence):
#     """
#     Truncates a power_series (an iterable of HybridisationTerm's)
#     leaving only orders up to truncation_power
#     """
#     return list(filter(lambda term: [pair[0] for pair in term.pairs] == sieve_sequence, power_series))

sieve_power = 3
terms = sieve_power_series(multitrap.deranged_determinant_by_order[multitrap.evaluation_order], sieve_power)
# sieve_sequence = ['x1', 'x2', 'x3', 'x4']
# terms = sieve_power_series_for_sequence(multitrap.deranged_determinant_by_order[multitrap.evaluation_order], sieve_sequence)

# terms = []
# for dim_j in list(multitrap.dimensions.values()):
#     for dim_k in list(multitrap.dimensions.values()):

#         if dim_j == dim_k:
#             continue

#         jn, km = dim_j.label, dim_k.label
#         Rjk = HybridisationTerm(sign=1, pairs=[(jn, km)])
#         print(term_string(Rjk))
#         terms.append(Rjk)

# NN = N * ax
# assert len(terms) == NN**2 - NN

assert len(terms) == 8

print(f'{len(terms)=}')
print(f'{terms=}')

ω_input = np.linspace(200, 350, 150, endpoint=True) * kHz

Rjk_fig = plt.figure(layout='constrained', figsize=(6, 12), dpi=200)
Rjk_ax_dict = Rjk_fig.subplot_mosaic([
    [term_string(term) for term in terms[0:2]],
    [term_string(term) for term in terms[2:4]],
    [term_string(term) for term in terms[4:6]],
    [term_string(term) for term in terms[6:8]],
])

marker = 'o'
colours = [
    "#FF5733", "#33FF57", "#3357FF", "#FF33A6", "#33FFF6",
    "#F6FF33", "#FF9D33", "#33FF9D", "#9D33FF", "#FF3333",
    "#33FF33", "#3333FF", "#FF5733", "#57FF33", "#5733FF",
    "#A6FF33", "#33A6FF", "#FF33F6", "#F633FF", "#33F6FF",
    "#9DFF33", "#FF339D", "#FF3339", "#39FF33", "#3339FF",
    "#33FFF9", "#FF33C1", "#C1FF33", "#33C1FF", "#FF9D57",
    "#57FF9D", "#9D57FF", "#FF579D", "#579DFF", "#33FF63",
    "#FF6333", "#6333FF", "#33FFA6", "#FFA633", "#A633FF",
    "#FF33FF", "#33FFFF", "#FFFF33", "#FF8C00", "#8C00FF",
    "#FF00A0", "#00A0FF", "#8CFF00", "#00FF8C", "#FF5732"
]
lim = 2
arrows = []
output_trails = []
trails = []
for ii, ax in enumerate(list(Rjk_ax_dict.values())):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal', 'box')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.grid(True, linestyle='--', alpha=0.7)

    arrows.append(ax.plot([], [], color=colours[ii], marker=marker))
    output_trails.append(ax.scatter([], [], color=colours[ii], alpha=0.5, s=10))
    trails.append([])

arrows = sum(arrows, [])

num_terms = len(terms)
arrows = arrows[:num_terms]
output_trails = output_trails[:num_terms]
trails = trails[:num_terms]
terms = terms[:num_terms]

# Animation function
def update(frame):
    ω = ω_input[frame]

    # for ii, term in enumerate(terms):
    for ii, ax in enumerate(list(Rjk_ax_dict.values())):
        term = terms[ii]
        fω = multitrap.hybridisation_term_value(term, ω)
        arrows[ii].set_label(f'{term_string(term)}(ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])
        ax.legend(loc='upper right')
        ax.set_title(f'ω={ω / kHz:.1f} kHz')

    return (Rjk_fig, *arrows, *output_trails)

# Create the animation
ani = FuncAnimation(Rjk_fig, update, frames=len(ω_input), interval=2, blit=False)

if MAKE_ANIMATIONS:
    save_animation(ani, f'grid of all Rjk (3)')
else:
    preview_animation(update, len(ω_input))

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


toy_params = {
    # the 2024 keys 'model': 'pairs', first_Ωx = 312 kHz and Ω_gap = 15 kHz, in today's names
    'Ω_model': 'pairs',
    'Ω_first': 312*kHz,
    'Ω_interpair_gap': 15*kHz,
    'Ω_intrapair_gap': 1.5*kHz,
    'Ωz_to_Ωx_multiplier': 10,

    'toy_gx': -40*kHz,
    'toy_gy': -40*kHz,
    # 'toy_gz': +170*kHz,
    # 'gxy': -2*kHz,
    # 'gxz': -15*kHz,
    # 'gyz': -15*kHz,

    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'L': 0.96*cm,
    'Finesse': 40_000,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
N, ax = 2, 2
ω_low, ω_high, ω_step = 0, 400, 0.5
multitrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=ax,
    ω_range=np.arange(ω_low, ω_high, ω_step) * kHz,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    # the 2×2 power-series pickle is tracked in git
    load_in_power_series=True,
)
multitrap.info
multitrap.info_per_dimension

ω_input = np.linspace(200, 350, 300, endpoint=True) * kHz

Det_fig = plt.figure(layout='constrained', figsize=(4, 4), dpi=150)
Det_ax_dict = Det_fig.subplot_mosaic([
    ['Det'],
])

marker = 'o'
colours = [
    "#FF5733", "#33FF57", "#3357FF", "#FF33A6", "#33FFF6",
    "#F6FF33", "#FF9D33", "#33FF9D", "#9D33FF", "#FF3333",
    "#33FF33", "#3333FF", "#FF5733", "#57FF33", "#5733FF",
    "#A6FF33", "#33A6FF", "#FF33F6", "#F633FF", "#33F6FF",
    "#9DFF33", "#FF339D", "#FF3339", "#39FF33", "#3339FF",
    "#33FFF9", "#FF33C1", "#C1FF33", "#33C1FF", "#FF9D57",
    "#57FF9D", "#9D57FF", "#FF579D", "#579DFF", "#33FF63",
    "#FF6333", "#6333FF", "#33FFA6", "#FFA633", "#A633FF",
    "#FF33FF", "#33FFFF", "#FFFF33", "#FF8C00", "#8C00FF",
    "#FF00A0", "#00A0FF", "#8CFF00", "#00FF8C", "#FF5732"
]
lim = 10
arrows = []
output_trails = []
trails = []
for ii, ax in enumerate(list(Det_ax_dict.values())):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal', 'box')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.grid(True, linestyle='--', alpha=0.7)

    arrows.append(ax.plot([], [], color=colours[ii], marker=marker))
    output_trails.append(ax.scatter([], [], color=colours[ii], alpha=0.5, s=10))
    trails.append([])

arrows = sum(arrows, [])

num_terms = len(terms)
arrows = arrows[:num_terms]
output_trails = output_trails[:num_terms]
trails = trails[:num_terms]
terms = terms[:num_terms]

# Animation function
def update(frame):
    ω = ω_input[frame]

    # for ii, term in enumerate(terms):
    for ii, ax in enumerate(list(Det_ax_dict.values())):
        term = terms[ii]
        fω = multitrap.deranged_determinant(ω)
        arrows[ii].set_label(f'Det(ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])
        ax.legend(loc='upper right')
        ax.set_title(f'ω={ω / kHz:.1f} kHz')

    return (Det_fig, *arrows, *output_trails)

# Create the animation
ani = FuncAnimation(Det_fig, update, frames=len(ω_input), interval=2, blit=False)

if MAKE_ANIMATIONS:
    save_animation(ani, f'deranged determinant')
else:
    preview_animation(update, len(ω_input))

# Diagonal adjugate $\text{adj}(\mathbf{D})_{x_1 x_1}$

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


toy_params = {
    # the 2024 keys 'model': 'pairs', first_Ωx = 312 kHz and Ω_gap = 15 kHz, in today's names
    'Ω_model': 'pairs',
    'Ω_first': 312*kHz,
    'Ω_interpair_gap': 15*kHz,
    'Ω_intrapair_gap': 1.5*kHz,
    'Ωz_to_Ωx_multiplier': 10,

    'toy_gx': -40*kHz,
    'toy_gy': -40*kHz,
    # 'toy_gz': +170*kHz,
    # 'gxy': -2*kHz,
    # 'gxz': -15*kHz,
    # 'gyz': -15*kHz,

    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'L': 0.96*cm,
    'Finesse': 40_000,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
N, ax = 2, 2
ω_low, ω_high, ω_step = 0, 400, 0.5
multitrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=ax,
    ω_range=np.arange(ω_low, ω_high, ω_step) * kHz,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    # the 2×2 power-series pickle is tracked in git
    load_in_power_series=True,
)
multitrap.info
multitrap.info_per_dimension

ω_input = np.linspace(200, 350, 300, endpoint=True) * kHz

Rjk_fig = plt.figure(layout='constrained', figsize=(4, 4), dpi=150)
Rjk_ax_dict = Rjk_fig.subplot_mosaic([
    ['Det'],
])

marker = 'o'
colours = [
    "#FF5733", "#33FF57", "#3357FF", "#FF33A6", "#33FFF6",
    "#F6FF33", "#FF9D33", "#33FF9D", "#9D33FF", "#FF3333",
    "#33FF33", "#3333FF", "#FF5733", "#57FF33", "#5733FF",
    "#A6FF33", "#33A6FF", "#FF33F6", "#F633FF", "#33F6FF",
    "#9DFF33", "#FF339D", "#FF3339", "#39FF33", "#3339FF",
    "#33FFF9", "#FF33C1", "#C1FF33", "#33C1FF", "#FF9D57",
    "#57FF9D", "#9D57FF", "#FF579D", "#579DFF", "#33FF63",
    "#FF6333", "#6333FF", "#33FFA6", "#FFA633", "#A633FF",
    "#FF33FF", "#33FFFF", "#FFFF33", "#FF8C00", "#8C00FF",
    "#FF00A0", "#00A0FF", "#8CFF00", "#00FF8C", "#FF5732"
]
lim = 10
arrows = []
output_trails = []
trails = []
for ii, ax in enumerate(list(Rjk_ax_dict.values())):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal', 'box')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.grid(True, linestyle='--', alpha=0.7)

    arrows.append(ax.plot([], [], color=colours[ii], marker=marker))
    output_trails.append(ax.scatter([], [], color=colours[ii], alpha=0.5, s=10))
    trails.append([])

arrows = sum(arrows, [])

num_terms = len(terms)
arrows = arrows[:num_terms]
output_trails = output_trails[:num_terms]
trails = trails[:num_terms]
terms = terms[:num_terms]

ij = 'x1x1'
# Animation function
def update(frame):
    ω = ω_input[frame]

    # for ii, term in enumerate(terms):
    for ii, ax in enumerate(list(Rjk_ax_dict.values())):
        term = terms[ii]
        fω = multitrap.deranged_adjugate_matrix_entry(ij, ω)
        arrows[ii].set_label(f'Adj[{ij}](ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])
        ax.legend(loc='upper right')
        ax.set_title(f'ω={ω / kHz:.1f} kHz')

    return (Rjk_fig, *arrows, *output_trails)

# Create the animation
ani = FuncAnimation(Rjk_fig, update, frames=len(ω_input), interval=2, blit=False)

if MAKE_ANIMATIONS:
    save_animation(ani, f'adjugate x1x1')
else:
    preview_animation(update, len(ω_input))

# Off-diagonal $\text{adj}(\mathbf{D})_{x_1 y_1}$

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


toy_params = {
    # the 2024 keys 'model': 'pairs', first_Ωx = 312 kHz and Ω_gap = 15 kHz, in today's names
    'Ω_model': 'pairs',
    'Ω_first': 312*kHz,
    'Ω_interpair_gap': 15*kHz,
    'Ω_intrapair_gap': 1.5*kHz,
    'Ωz_to_Ωx_multiplier': 10,

    'toy_gx': -40*kHz,
    'toy_gy': -40*kHz,
    # 'toy_gz': +170*kHz,
    # 'gxy': -2*kHz,
    # 'gxz': -15*kHz,
    # 'gyz': -15*kHz,

    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'L': 0.96*cm,
    'Finesse': 40_000,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
N, ax = 2, 2
ω_low, ω_high, ω_step = 0, 400, 0.5
multitrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=ax,
    ω_range=np.arange(ω_low, ω_high, ω_step) * kHz,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    # the 2×2 power-series pickle is tracked in git
    load_in_power_series=True,
)
multitrap.info
multitrap.info_per_dimension

ω_input = np.linspace(200, 350, 300, endpoint=True) * kHz

Rjk_fig = plt.figure(layout='constrained', figsize=(4, 4), dpi=150)
Rjk_ax_dict = Rjk_fig.subplot_mosaic([
    ['Det'],
])

marker = 'o'
colours = [
    "#FF5733", "#33FF57", "#3357FF", "#FF33A6", "#33FFF6",
    "#F6FF33", "#FF9D33", "#33FF9D", "#9D33FF", "#FF3333",
    "#33FF33", "#3333FF", "#FF5733", "#57FF33", "#5733FF",
    "#A6FF33", "#33A6FF", "#FF33F6", "#F633FF", "#33F6FF",
    "#9DFF33", "#FF339D", "#FF3339", "#39FF33", "#3339FF",
    "#33FFF9", "#FF33C1", "#C1FF33", "#33C1FF", "#FF9D57",
    "#57FF9D", "#9D57FF", "#FF579D", "#579DFF", "#33FF63",
    "#FF6333", "#6333FF", "#33FFA6", "#FFA633", "#A633FF",
    "#FF33FF", "#33FFFF", "#FFFF33", "#FF8C00", "#8C00FF",
    "#FF00A0", "#00A0FF", "#8CFF00", "#00FF8C", "#FF5732"
]
lim = 10
arrows = []
output_trails = []
trails = []
for ii, ax in enumerate(list(Rjk_ax_dict.values())):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal', 'box')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.grid(True, linestyle='--', alpha=0.7)

    arrows.append(ax.plot([], [], color=colours[ii], marker=marker))
    output_trails.append(ax.scatter([], [], color=colours[ii], alpha=0.5, s=10))
    trails.append([])

arrows = sum(arrows, [])

num_terms = len(terms)
arrows = arrows[:num_terms]
output_trails = output_trails[:num_terms]
trails = trails[:num_terms]
terms = terms[:num_terms]

ij = 'x1y1'
# Animation function
def update(frame):
    ω = ω_input[frame]

    # for ii, term in enumerate(terms):
    for ii, ax in enumerate(list(Rjk_ax_dict.values())):
        term = terms[ii]
        fω = multitrap.deranged_adjugate_matrix_entry(ij, ω)
        arrows[ii].set_label(f'Adj[{ij}](ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])
        ax.legend(loc='upper right')
        ax.set_title(f'ω={ω / kHz:.1f} kHz')

    return (Rjk_fig, *arrows, *output_trails)

# Create the animation
ani = FuncAnimation(Rjk_fig, update, frames=len(ω_input), interval=2, blit=False)

if MAKE_ANIMATIONS:
    save_animation(ani, f'adjugate x1y1')
else:
    preview_animation(update, len(ω_input))

# Grid: adjugate

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


toy_params = {
    # the 2024 keys 'model': 'pairs', first_Ωx = 312 kHz and Ω_gap = 15 kHz, in today's names
    'Ω_model': 'pairs',
    'Ω_first': 312*kHz,
    'Ω_interpair_gap': 15*kHz,
    'Ω_intrapair_gap': 1.5*kHz,
    'Ωz_to_Ωx_multiplier': 10,

    'toy_gx': -40*kHz,
    'toy_gy': -40*kHz,
    # 'toy_gz': +170*kHz,
    # 'gxy': -2*kHz,
    # 'gxz': -15*kHz,
    # 'gyz': -15*kHz,

    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'L': 0.96*cm,
    'Finesse': 40_000,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
N, ax = 2, 2
ω_low, ω_high, ω_step = 0, 400, 0.5
multitrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=ax,
    ω_range=np.arange(ω_low, ω_high, ω_step) * kHz,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    # the 2×2 power-series pickle is tracked in git
    load_in_power_series=True,
)
multitrap.info
multitrap.info_per_dimension

ω_input = np.linspace(250, 350, 200, endpoint=True) * kHz

all_ij = []
for dim_j in list(multitrap.dimensions.values()):
    for dim_k in list(multitrap.dimensions.values()):

        jn, km = dim_j.label, dim_k.label
        all_ij.append(f'{jn}{km}')


fig = plt.figure(layout='constrained', figsize=(7, 7), dpi=100)
ax_dict = fig.subplot_mosaic([
    [ij for ij in all_ij[0:4]],
    [ij for ij in all_ij[4:8]],
    [ij for ij in all_ij[8:12]],
    [ij for ij in all_ij[12:16]],
])

marker = 'o'
colours = [
    "#FF5733", "#33FF57", "#3357FF", "#FF33A6", "#33FFF6",
    "#F6FF33", "#FF9D33", "#33FF9D", "#9D33FF", "#FF3333",
    "#33FF33", "#3333FF", "#FF5733", "#57FF33", "#5733FF",
    "#A6FF33", "#33A6FF", "#FF33F6", "#F633FF", "#33F6FF",
    "#9DFF33", "#FF339D", "#FF3339", "#39FF33", "#3339FF",
    "#33FFF9", "#FF33C1", "#C1FF33", "#33C1FF", "#FF9D57",
    "#57FF9D", "#9D57FF", "#FF579D", "#579DFF", "#33FF63",
    "#FF6333", "#6333FF", "#33FFA6", "#FFA633", "#A633FF",
    "#FF33FF", "#33FFFF", "#FFFF33", "#FF8C00", "#8C00FF",
    "#FF00A0", "#00A0FF", "#8CFF00", "#00FF8C", "#FF5732"
]
lim = 5
arrows = []
output_trails = []
trails = []
for ii, ax in enumerate(list(ax_dict.values())):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal', 'box')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.grid(True, linestyle='--', alpha=0.7)

    arrows.append(ax.plot([], [], color=colours[ii], marker=marker))
    output_trails.append(ax.scatter([], [], color=colours[ii], alpha=0.5, s=10))
    trails.append([])

arrows = sum(arrows, [])

# Animation function
# the original notebook relied on `det` from the 'Grid: adjugate' cell below having been run first
det = np.array([multitrap.deranged_determinant(ω) for ω in ω_input])

def update(frame):
    ω = ω_input[frame]
    det_ω = det[frame]
    for ii, (ij, ax) in enumerate(list(ax_dict.items())):
        fω = multitrap.deranged_adjugate_matrix_entry(ij, ω)
        arrows[ii].set_label(f'{ij}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])
        ax.legend(loc='upper right')
        ax.set_title(f'{fω:.3f}')

    fig.suptitle(f'adj(ω); ω={ω / kHz:.1f} kHz')
    return (fig, *arrows, *output_trails)

# Create the animation
ani = FuncAnimation(fig, update, frames=len(ω_input), interval=2, blit=False)

if MAKE_ANIMATIONS:
    save_animation(ani, f'adjugate grid')
else:
    preview_animation(update, len(ω_input))

# Grid: All solutions $h_{jk}$

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


toy_params = {
    # the 2024 keys 'model': 'pairs', first_Ωx = 312 kHz and Ω_gap = 15 kHz, in today's names
    'Ω_model': 'pairs',
    'Ω_first': 312*kHz,
    'Ω_interpair_gap': 15*kHz,
    'Ω_intrapair_gap': 1.5*kHz,
    'Ωz_to_Ωx_multiplier': 10,

    'toy_gx': -40*kHz,
    'toy_gy': -40*kHz,
    # 'toy_gz': +170*kHz,
    # 'gxy': -2*kHz,
    # 'gxz': -15*kHz,
    # 'gyz': -15*kHz,

    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'L': 0.96*cm,
    'Finesse': 40_000,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
N, ax = 2, 2
ω_low, ω_high, ω_step = 0, 400, 0.5
multitrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=ax,
    ω_range=np.arange(ω_low, ω_high, ω_step) * kHz,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    # the 2×2 power-series pickle is tracked in git
    load_in_power_series=True,
)
multitrap.info
multitrap.info_per_dimension

ω_input = np.linspace(295, 315, 400, endpoint=True) * kHz

all_ij = []
for dim_j in list(multitrap.dimensions.values()):
    for dim_k in list(multitrap.dimensions.values()):

        jn, km = dim_j.label, dim_k.label
        all_ij.append(f'{jn}{km}')


hjk_fig = plt.figure(layout='constrained', figsize=(8, 8), dpi=125)
hjk_ax_dict = hjk_fig.subplot_mosaic([
    [ij for ij in all_ij[0:4]],
    [ij for ij in all_ij[4:8]],
    [ij for ij in all_ij[8:12]],
    [ij for ij in all_ij[12:16]],
])

marker = 'o'
colours = [
    "#FF5733", "#33FF57", "#3357FF", "#FF33A6", "#33FFF6",
    "#F6FF33", "#FF9D33", "#33FF9D", "#9D33FF", "#FF3333",
    "#33FF33", "#3333FF", "#FF5733", "#57FF33", "#5733FF",
    "#A6FF33", "#33A6FF", "#FF33F6", "#F633FF", "#33F6FF",
    "#9DFF33", "#FF339D", "#FF3339", "#39FF33", "#3339FF",
    "#33FFF9", "#FF33C1", "#C1FF33", "#33C1FF", "#FF9D57",
    "#57FF9D", "#9D57FF", "#FF579D", "#579DFF", "#33FF63",
    "#FF6333", "#6333FF", "#33FFA6", "#FFA633", "#A633FF",
    "#FF33FF", "#33FFFF", "#FFFF33", "#FF8C00", "#8C00FF",
    "#FF00A0", "#00A0FF", "#8CFF00", "#00FF8C", "#FF5732"
]
lim = 50
arrows = []
output_trails = []
trails = []
for ii, ax in enumerate(list(hjk_ax_dict.values())):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal', 'box')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.grid(True, linestyle='--', alpha=0.7)

    arrows.append(ax.plot([], [], color=colours[ii], marker=marker))
    output_trails.append(ax.scatter([], [], color=colours[ii], alpha=0.5, s=10))
    trails.append([])

arrows = sum(arrows, [])

det = np.array([multitrap.deranged_determinant(ω) for ω in ω_input])

# Animation function
def update(frame):
    ω = ω_input[frame]
    det_ω = det[frame]
    for ii, (ij, ax) in enumerate(list(hjk_ax_dict.items())):
        # fω = multitrap.deranged_adjugate_matrix_entry(ij, ω) / multitrap.deranged_determinant(ω)
        fω = multitrap.deranged_adjugate_matrix_entry(ij, ω) / det_ω
        # arrows[ii].set_label(f'h{ij}(ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])
        # ax.legend(loc='upper right')
        ax.set_title(f'h{ij}(ω)={fω:.3f}')

    hjk_fig.suptitle(f'ω={ω / kHz:.1f} kHz')
    return (hjk_fig, *arrows, *output_trails)

# Create the animation
ani = FuncAnimation(hjk_fig, update, frames=len(ω_input), interval=2, blit=False)

if MAKE_ANIMATIONS:
    save_animation(ani, f'all solutions hjk')
else:
    preview_animation(update, len(ω_input))